In [14]:
import os
import re

# 日本語の正規表現（ひらがな、カタカナ、漢字）
japanese_pattern = re.compile(r'[\u3040-\u30FF\u4E00-\u9FFF]')

# コメント検出用パターン
comment_line_pattern = re.compile(r'^\s*[#＃]')
docstring_pattern = re.compile(r'(["\']{3})(.*?)(\1)', re.DOTALL)

def contains_japanese(text):
    return japanese_pattern.search(text) is not None

def scan_file_for_japanese_comments(file_path):
    matches = []
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    in_docstring = False
    docstring_delim = None

    for i, line in enumerate(lines):
        stripped = line.strip()

        # コメント行の中に日本語
        if comment_line_pattern.match(stripped) and contains_japanese(line):
            matches.append((i + 1, line.strip()))
            continue

        # ブロックdocstringの中に日本語があるか
        if not in_docstring and (stripped.startswith('"""') or stripped.startswith("'''")):
            docstring_delim = stripped[:3]
            in_docstring = True
            if contains_japanese(line):
                matches.append((i + 1, line.strip()))
            continue
        elif in_docstring:
            if contains_japanese(line):
                matches.append((i + 1, line.strip()))
            if docstring_delim in stripped:
                in_docstring = False
            continue

    return matches

def scan_directory(path):
    for root, _, files in os.walk(path):
        for fname in files:
            if fname.endswith('.py'):
                fpath = os.path.join(root, fname)
                matches = scan_file_for_japanese_comments(fpath)
                if matches:
                    print(f"\n📝 {fpath} に日本語コメントがあります:")
                    if "a2c" in fpath:
                        for lineno, content in matches:
                            print(f"  Line {lineno}: {content}")

# 実行対象のルートディレクトリを指定
workspace_path = "/home/mil/shitanda/CPO_NeurIPS2025/supplementary_code"  # 必要に応じて変更
scan_directory(workspace_path)
